# GeoMamba — LR TARAMASI: (d) bulgusunun geçerlilik tehdidini kapat (Kaggle T4)

`report/fullgeo_sonuc_2026-08-17.md` şunu buldu: eşdeğişken model 1-adım
tahmininde her veri miktarında 5/5 seed kazanıyor, kazanç 20×→3.9× daralıyor.

**Ama tek bir öğrenme oranıyla (3e-4) ölçüldü.** Eşdeğişken modeller o ayarı
baseline'dan daha çok seviyor olabilir. Bu tarama her kolu **kendi en iyi
lr'sinde** koşturur; karşılaştırma en-iyi-vs-en-iyi olur.

**Metodolojik kural:** lr seçimi `best_val_mse` ile yapılır, `test_mse` ile
**asla**. Test setine bakarak hiperparametre seçmek sonucu geçersiz kılar.

3 kol (`mamba`, `fullgeonb`, `fullgeo`) × 4 lr (1e-4, 3e-4, 1e-3, 3e-3)
× 2 N (30, 300) × 3 seed = **72 koşu**, T4×2'de ~2.5 saat.

İki N birlikte, çünkü iddia farkın *büyüklüğü* değil, N ile **daralması**;
tek N'de tararsak daralmanın lr artefaktı olmadığını gösteremeyiz.

**Kurulum:** Session options → GPU **T4 x2**, Internet **On**; Add-ons →
Secrets → `GH_TOKEN`. Sonra **Save Version → Save & Run All (Commit)**.


In [ ]:
%cd /kaggle/working
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GH_TOKEN")
!rm -rf /kaggle/working/geomamba
!git clone --depth 1 https://{token}@github.com/yusuftiryaki/geomamba.git /kaggle/working/geomamba


In [ ]:
%cd /kaggle/working/geomamba
import os, torch
os.environ["PYTHONPATH"] = "/kaggle/working/geomamba/src"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "YOK - Session options'tan GPU secin!")


In [ ]:
import os, subprocess, sys, pathlib
GRID = "configs/grid/m7_lr_sweep.yaml"

n_gpu = int(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU "))
print("GPU sayisi:", n_gpu)

# datasets once, then one process per GPU
subprocess.run([sys.executable, "scripts/run_grid.py", "--grid", GRID, "--data-only"], check=True)
procs = []
for k in range(max(n_gpu, 1)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(f"shard{k}.log", "w")
    procs.append((k, log, subprocess.Popen(
        [sys.executable, "scripts/run_grid.py", "--grid", GRID, "--skip-data",
         "--shard", f"{k}/{max(n_gpu, 1)}"],
        env=env, stdout=log, stderr=subprocess.STDOUT)))
print(f"{len(procs)} shard baslatildi; loglar shard*.log")
for k, log, p in procs:
    rc = p.wait(); log.close()
    print(f"shard {k} bitti, rc={rc}")
    print(pathlib.Path(f"shard{k}.log").read_text()[-2000:])


In [ ]:
import glob, shutil, pandas as pd
# Shard sonuçları iki yere yazılır: koşu sırasında artımlı olarak
# /kaggle/working/grid_results-shard*.csv, shard TAMAMEN bitince de
# experiments/ altına. Shard zaman aşımına uğrarsa ikincisi hiç yazılmaz --
# o yüzden ikisine de bakıyoruz, yoksa tamamlanmış koşular kayıp görünür.
paths = sorted(set(glob.glob("experiments/grid_results*.csv")
                   + glob.glob("/kaggle/working/grid_results-shard*.csv")))
frames = []
for f in paths:
    try:
        frames.append(pd.read_csv(f))
    except Exception as e:
        print("okunamadi", f, e)
if frames:
    df = pd.concat(frames, ignore_index=True).drop_duplicates(subset="name", keep="last")
    df.to_csv("/kaggle/working/grid_results.csv", index=False)
    for f in glob.glob("experiments/grid_results*.json"):
        shutil.copy(f, "/kaggle/working/")
    print(f"{len(df)} kosu toplandi ({len(paths)} dosyadan)")
    display(df)
else:
    print("sonuc yok - shard*.log dosyalarina bakin")
    for f in sorted(glob.glob("shard*.log")):
        print("=" * 20, f)
        print(open(f).read()[-3000:])


In [ ]:
import os, pandas as pd
if not os.path.exists("/kaggle/working/grid_results.csv"):
    raise SystemExit("Sonuc dosyasi yok - onceki hucrenin ciktisina bakin.")
df = pd.read_csv("/kaggle/working/grid_results.csv")
p = df.name.str.split("-")
df["arm"] = p.str[1]; df["n"] = p.str[2].str[1:].astype(int); df["lr"] = p.str[3]

LR = {"1e4": 1e-4, "3e4": 3e-4, "1e3": 1e-3, "3e3": 3e-3}
df["lr_val"] = df.lr.map(LR)

print("=== 1) lr secimi: VALIDATION hatasi (test'e BAKMADAN) ===")
val = df.pivot_table(index=["n", "arm"], columns="lr_val", values="best_val_mse", aggfunc="median")
display(val)
best = val.idxmin(axis=1).rename("en_iyi_lr")
print("\nHer kolun kendi en iyi lr'si (validation'a gore):")
display(best)

print("\n=== 2) SIMDI test: her kol kendi en iyi lr'sinde ===")
rows = []
for (n, arm), lr_star in best.items():
    sel = df[(df.n == n) & (df.arm == arm) & (df.lr_val == lr_star)]
    rows.append(dict(n=n, arm=arm, lr=lr_star, test_mse=sel.test_mse.median(),
                     h100=sel.rollout_mse_h100.median(), rot_gap=sel.rot_gap.mean(),
                     n_seed=len(sel)))
best_tbl = pd.DataFrame(rows).pivot_table(index="n", columns="arm",
                                          values=["lr", "test_mse", "rot_gap"])
display(best_tbl)

print("\n=== 3) KARAR: en-iyi-vs-en-iyi kazanc orani ===")
for n in sorted(df.n.unique()):
    b = {r["arm"]: r for r in rows if r["n"] == n}
    if "mamba" in b:
        for a in ("fullgeonb", "fullgeo"):
            if a in b:
                print(f"  N={n:<4} {a:<10} {b['mamba']['test_mse']/b[a]['test_mse']:6.2f}x "
                      f"(lr {b[a]['lr']:.0e} vs baseline lr {b['mamba']['lr']:.0e})")
print("\nBEKLENTI: N=30'da oran N=300'dekinden BUYUK kalmali (daralan fark).")
print("Oran ~1'e duserse bulgu lr artefaktiydi -> proje negatif sonucla kapanir.")
